# Quantitative Researcher API

WorldQuant-style workflow (decision note DEC-017): seed -> single-alpha evaluation -> GA mining from passing seeds -> pool delivery.

In [ ]:
from quant_api.core import DataConfig
from quant_api.alpha import (
    AlphaConfig, GateCriteria, validate_seed, evaluate_seed,
    mine_seeds, screen_batch, build_spec_sheet, deliver_to_pool, score,
)

# Data is auto-resolved from the repo research catalog; point elsewhere with
# DataConfig(catalog_path="/path/to/catalog", start=..., end=...).
WINDOW = DataConfig(start="2026-07-15", end="2026-08-28")
cfg = AlphaConfig(data=WINDOW, ga_population_size=8, ga_generations=2, ga_seed=7)
print("ok")

In [ ]:
seed = "close - ewma(close, 8)"
canonical = validate_seed(seed)
tear = evaluate_seed(seed, cfg)
print(canonical, "->", tear.verdict, tear.reasons)
print({k: tear.metrics[k] for k in ("net_sharpe", "best_abs_ic", "cost_drag_pct")})

In [ ]:
# Breed only from evaluation-passing seeds (WorldQuant-style).
passing = [s for s in ("close - ewma(close, 8)", "ts_returns(close, 8)")
           if evaluate_seed(s, cfg).verdict == "IN"]
if not passing:
    # The default gate is strict on this 1-month window; fall back to the
    # raw seed so the GA demo still runs (a real user would only breed passes).
    passing = ["ts_returns(close, 8)"]
bred = mine_seeds(passing, cfg)
print("passing seeds:", passing)
print("bred candidates:", bred[:3])

In [ ]:
import tempfile as _tf
trial_root = _tf.mkdtemp(prefix="trial_ledger_")
funnel = screen_batch(bred[:4], cfg, record_trials=True, trial_root=trial_root, source="ga")
print(funnel["funnel"])
print("trial ledger lines:", len(open(trial_root + "/trial_ledger.jsonl").readlines()))

In [ ]:
import tempfile
pool_dir = tempfile.mkdtemp(prefix="research_pool_")
survivor = funnel["tear_sheets"][0]
if survivor.verdict == "IN":
    entry = deliver_to_pool(survivor, build_spec_sheet(survivor, config=cfg),
                            source="ga", pool_root=pool_dir)
    print("delivered:", entry.alpha_id, "->", pool_dir)
    # Explicit on-demand export of the tear sheet (reports are never auto-saved).
    out = survivor.to_json(pool_dir + "/tearsheet.json")
    print("tear sheet exported:", out)
else:
    print("no IN survivor in this batch:", survivor.reasons)